# Feature Engineering
## Project: Real-Time Anomaly Detection MLOps Pipeline
**Input:** `data/processed/selected_series.parquet` — 38 series, 270,723 points, 5-minute intervals

**Notebook goals:**
- Load selected series into DuckDB as a queryable table
- Compute rolling features: mean, std, z-score, rate of change, lag features
- Design and validate sliding window parameters
- Normalize each series independently
- Produce reproducible train/val/test splits stratified by series
- Write final feature table to DuckDB for consumption by Notebook 03

*Personal portfolio project — not affiliated with the University of Toronto.*

In [1]:
# everything we need for feature engineering
import os
import json
from pathlib import Path

import pandas as pd
import numpy as np
import duckdb
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler

plt.rcParams["figure.figsize"] = (14, 4)
plt.rcParams["figure.dpi"] = 100
sns.set_theme(style="darkgrid")

print("imports done")

imports done


## 1. Load Selected Series into DuckDB

In [2]:
# create a persistent DuckDB database file in the processed folder
# this becomes our local feature store for the rest of the project
db_path = "../data/processed/feature_store.duckdb"
con = duckdb.connect(db_path)

# load the parquet file directly into a DuckDB table
# DuckDB can read parquet natively without loading into pandas first
con.execute("""
    CREATE OR REPLACE TABLE raw_series AS
    SELECT * FROM read_parquet('../data/processed/selected_series.parquet')
""")

# quick sanity check
result = con.execute("""
    SELECT
        category,
        COUNT(DISTINCT series_name) AS n_series,
        COUNT(*)                    AS n_points,
        MIN(timestamp)              AS earliest,
        MAX(timestamp)              AS latest
    FROM raw_series
    GROUP BY category
    ORDER BY category
""").df()

print(result.to_string(index=False))

             category  n_series  n_points            earliest              latest
  artificialNoAnomaly         5     20160 2014-04-01 00:00:00 2014-04-14 23:55:00
artificialWithAnomaly         6     24192 2014-04-01 00:00:00 2014-04-14 23:55:00
    realAWSCloudwatch        17     67740 2013-10-09 16:25:00 2014-04-24 00:39:00
           realTweets        10    158631 2015-02-26 21:42:53 2015-04-23 02:47:53


## 2. Compute Rolling Features

In [3]:
# load raw series into pandas for feature computation
# we compute all rolling features per series independently
# so one series's stats don't bleed into another
raw_df = con.execute("SELECT * FROM raw_series ORDER BY series_name, timestamp").df()

# container for all series with features added
featured_rows = []

for series_name, group in raw_df.groupby("series_name"):
    g = group.sort_values("timestamp").copy()
    
    # rolling window of 12 steps = 1 hour of 5-min data
    # min_periods=1 so we don't lose the first 11 rows
    roll = g["value"].rolling(window=12, min_periods=1)
    
    # rolling stats
    g["rolling_mean"]   = roll.mean()
    g["rolling_std"]    = roll.std().fillna(0)
    
    # z-score: how many stds away from the local rolling mean
    g["rolling_zscore"] = (
        (g["value"] - g["rolling_mean"]) / (g["rolling_std"] + 1e-8)
    )
    
    # rate of change: how much the value moved since last step
    g["rate_of_change"] = g["value"].diff().fillna(0)
    
    # lag features: value 1 and 2 steps back
    g["lag_1"] = g["value"].shift(1).fillna(method="bfill")
    g["lag_2"] = g["value"].shift(2).fillna(method="bfill")
    
    featured_rows.append(g)

featured_df = pd.concat(featured_rows, ignore_index=True)

print(f"Featured dataframe shape: {featured_df.shape}")
print(f"\nColumns: {list(featured_df.columns)}")
print(f"\nSample rows:")
featured_df.head(3)

Featured dataframe shape: (270723, 11)

Columns: ['timestamp', 'value', 'category', 'series_name', 'is_anomaly', 'rolling_mean', 'rolling_std', 'rolling_zscore', 'rate_of_change', 'lag_1', 'lag_2']

Sample rows:


C:\Users\nabee\AppData\Local\Temp\ipykernel_17632\1772750521.py:29: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  g["lag_1"] = g["value"].shift(1).fillna(method="bfill")
C:\Users\nabee\AppData\Local\Temp\ipykernel_17632\1772750521.py:30: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  g["lag_2"] = g["value"].shift(2).fillna(method="bfill")
C:\Users\nabee\AppData\Local\Temp\ipykernel_17632\1772750521.py:29: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  g["lag_1"] = g["value"].shift(1).fillna(method="bfill")
C:\Users\nabee\AppData\Local\Temp\ipykernel_17632\1772750521.py:30: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  g["lag_2"] = g["valu

,timestamp,value,category,series_name,is_anomaly,rolling_mean,rolling_std,rolling_zscore,rate_of_change,lag_1,lag_2
0,2015-02-26 21:42:53,104.0,realTweets,Twitter_volume_AAPL,0,104.0,0.000000,0.000000,0.0,104.0,104.0
1,2015-02-26 21:47:53,100.0,realTweets,Twitter_volume_AAPL,0,102.0,2.828427,-0.707107,-4.0,104.0,104.0
2,2015-02-26 21:52:53,99.0,realTweets,Twitter_volume_AAPL,0,101.0,2.645751,-0.755929,-1.0,100.0,104.0


## 3. Normalize Each Series Independently

In [4]:
# normalize each series independently using min-max scaling
# this keeps each series in [0, 1] regardless of its original value range
# we scale the feature columns only — not timestamps, labels, or identifiers

feature_cols = [
    "value",
    "rolling_mean",
    "rolling_std",
    "rolling_zscore",
    "rate_of_change",
    "lag_1",
    "lag_2"
]

normalized_rows = []

for series_name, group in featured_df.groupby("series_name"):
    g = group.copy()
    
    scaler = MinMaxScaler()
    g[feature_cols] = scaler.fit_transform(g[feature_cols])
    
    normalized_rows.append(g)

normalized_df = pd.concat(normalized_rows, ignore_index=True)

# sanity check: every feature column should now be in [0, 1]
print("Feature value ranges after normalization:")
print(normalized_df[feature_cols].agg(["min", "max"]).round(4).to_string())

Feature value ranges after normalization:
     value  rolling_mean  rolling_std  rolling_zscore  rate_of_change  lag_1  lag_2
min    0.0           0.0          0.0             0.0             0.0    0.0    0.0
max    1.0           1.0          1.0             1.0             1.0    1.0    1.0


## 4. Sliding Window Design and Validation

In [5]:
# sliding window parameters
# window_size = 12 steps = 1 hour of 5-min data (what the LSTM sees at once)
# stride = 1 step = 5 minutes (how far we slide forward each time)
WINDOW_SIZE = 12
STRIDE = 1

# count how many windows we'll get per series
window_counts = []

for series_name, group in normalized_df.groupby("series_name"):
    n_points = len(group)
    n_windows = (n_points - WINDOW_SIZE) // STRIDE + 1
    n_anomaly_windows = 0
    
    # a window is anomalous if ANY point inside it is anomalous
    g = group.sort_values("timestamp").reset_index(drop=True)
    for i in range(0, n_points - WINDOW_SIZE + 1, STRIDE):
        if g["is_anomaly"].iloc[i : i + WINDOW_SIZE].any():
            n_anomaly_windows += 1
    
    window_counts.append({
        "series_name":       series_name,
        "category":          group["category"].iloc[0],
        "n_points":          n_points,
        "n_windows":         n_windows,
        "n_anomaly_windows": n_anomaly_windows,
        "anomaly_rate":      round(100 * n_anomaly_windows / n_windows, 2)
    })

window_df = pd.DataFrame(window_counts)

print(f"Total windows across all series: {window_df['n_windows'].sum():,}")
print(f"Total anomaly windows: {window_df['n_anomaly_windows'].sum():,}")
print(f"Overall window anomaly rate: {100 * window_df['n_anomaly_windows'].sum() / window_df['n_windows'].sum():.2f}%")
print(f"\nPer category summary:")
print(
    window_df.groupby("category")[["n_windows", "n_anomaly_windows"]]
    .sum()
    .assign(anomaly_rate=lambda x: (100 * x["n_anomaly_windows"] / x["n_windows"]).round(2))
    .to_string()
)

Total windows across all series: 270,305
Total anomaly windows: 25,140
Overall window anomaly rate: 9.30%

Per category summary:
                       n_windows  n_anomaly_windows  anomaly_rate
category                                                         
artificialNoAnomaly        20105                  0          0.00
artificialWithAnomaly      24126               2484         10.30
realAWSCloudwatch          67553               6642          9.83
realTweets                158521              16014         10.10


## 5. Train / Validation / Test Splits

In [6]:
# split strategy: 70% train, 15% val, 15% test
# we split each series chronologically (no shuffling — time series order matters)
# then assign series to splits so every category is represented in all three splits

TRAIN_RATIO = 0.70
VAL_RATIO   = 0.15
# test gets the remainder (0.15)

split_rows = []

for series_name, group in normalized_df.groupby("series_name"):
    g = group.sort_values("timestamp").reset_index(drop=True)
    n = len(g)
    
    train_end = int(n * TRAIN_RATIO)
    val_end   = int(n * (TRAIN_RATIO + VAL_RATIO))
    
    g["split"] = "test"
    g.loc[:train_end - 1, "split"] = "train"
    g.loc[train_end:val_end - 1, "split"] = "val"
    
    split_rows.append(g)

split_df = pd.concat(split_rows, ignore_index=True)

# summary of split sizes
print("Split sizes:")
split_summary = (
    split_df.groupby("split")
    .agg(
        n_points    = ("value", "count"),
        n_anomalies = ("is_anomaly", "sum")
    )
    .assign(anomaly_rate=lambda x: (100 * x["n_anomalies"] / x["n_points"]).round(2))
)
print(split_summary.to_string())

print(f"\nSplit sizes by category:")
print(
    split_df.groupby(["category", "split"])["value"]
    .count()
    .unstack(fill_value=0)
    .to_string()
)

Split sizes:
       n_points  n_anomalies  anomaly_rate
split                                     
test      40619         2466          6.07
train    189492        17744          9.36
val       40612         4171         10.27

Split sizes by category:
split                   test   train    val
category                                   
artificialNoAnomaly     3025   14110   3025
artificialWithAnomaly   3630   16932   3630
realAWSCloudwatch      10166   47412  10162
realTweets             23798  111038  23795


## 6. Write Feature Table to DuckDB

In [7]:
# write the final feature table to DuckDB
# this is what Notebook 03 will load directly for LSTM training
con.execute("""
    CREATE OR REPLACE TABLE features AS
    SELECT * FROM split_df
""")

# verify the write
verify = con.execute("""
    SELECT
        split,
        COUNT(*)                    AS n_points,
        COUNT(DISTINCT series_name) AS n_series,
        ROUND(100.0 * SUM(is_anomaly) / COUNT(*), 2) AS anomaly_rate
    FROM features
    GROUP BY split
    ORDER BY split
""").df()

print("DuckDB features table verified:")
print(verify.to_string(index=False))

# also save as parquet for portability
split_df.to_parquet("../data/processed/features.parquet", index=False)
print(f"\nParquet saved: {os.path.getsize('../data/processed/features.parquet') / 1024:.1f} KB")

# close the connection cleanly
con.close()
print("DuckDB connection closed")

DuckDB features table verified:
split  n_points  n_series  anomaly_rate
 test     40619        38          6.07
train    189492        38          9.36
  val     40612        38         10.27

Parquet saved: 8332.6 KB
DuckDB connection closed


## Summary

### What we built
- Loaded 38 selected series into DuckDB as a queryable feature store
- Computed 6 rolling features per series: rolling mean, rolling std, z-score, rate of change, lag-1, lag-2
- Rolling window: 12 steps = 1 hour of 5-minute data
- Normalized all feature columns independently per series using min-max scaling → all values in [0, 1]
- Designed sliding windows: 270,305 total windows, 9.3% anomaly rate, stride = 1 step

### Split design
| Split | Points | Series | Anomaly rate |
|-------|--------|--------|--------------|
| Train | 189,492 | 38 | 9.36% |
| Val   | 40,612  | 38 | 10.27% |
| Test  | 40,619  | 38 | 6.07% |

Splits are chronological per series — no data leakage between train and test.

### Outputs saved
- `data/processed/features.parquet` — normalized features with split labels (8.3 MB)
- `data/processed/feature_store.duckdb` — queryable DuckDB database with `raw_series` and `features` tables

### Next: Notebook 03 — LSTM Autoencoder
- Load `features.parquet` train split
- Build PyTorch Dataset and DataLoader for sliding windows
- Define LSTM Autoencoder architecture (encoder → bottleneck → decoder)
- Train on normal segments only — anomaly labels withheld during training
- Log all experiments to MLflow: hyperparameters, loss curves, model artifacts
- Evaluate reconstruction error on val and test splits